# 实践 4.2  从PLINK质控到rMVP输入：基因型数据迁移链

《统计遗传与智能育种学》第四章 · 电子资源 `04b_gwas_toolchain.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：04b_gwas_toolchain.ipynb。实践构建PLINK基因型质控—Beagle填充—格式转换—rMVP读取与GWAS的数据迁移链：生成小型VCF，在Python中复现样本/位点缺失率、MAF与Hardy–Weinberg检查的含义，模拟缺失基因型并用简单均值填充作为反例，再写出剂量表与表型文件；同时保留PLINK 2.0、Beagle和rMVP的真实流程模板。在近交、自交系或结构明显的育种群体中，不宜用Hardy–Weinberg检验统一过滤位点。

Notebook给出与其自产生demo.vcf和demo_trait.tsv相对应的rMVP读取与混合线性模型（mixed linear model, MLM）运行模板，Python核心示例不执行这些外部命令。群体结构造成的关联混杂由实践4.1演示，功效与I类错误由实践4.3演示；这样每个Notebook只承担它实际运行的计算任务。正式分析应固定PLINK、Beagle与rMVP版本，并按所用版本的官方文档核对接口。

流程映射：PLINK质控 → Beagle填充 → rMVP数据读取/GWAS。  
电子版：04b_gwas_toolchain.ipynb；内置虚拟VCF、表型与剂量文件用于Python核心示例；外部工具命令作为迁移到真实流程的版本化模板保留。

## 输入、完整代码、实际输出与解释

输入与可复现条件：80个样本的小型VCF与表型；SEED=402；Python核心无需PLINK/Beagle/R。Markdown中的PLINK 2.0、Beagle和rMVP命令是迁移到真实数据的正式分析工具模板。

电子版Markdown中保留的外部工具代码模板（不作为当前Python核心的已执行输出）

**bash代码**（外部工具代码模板，本 notebook 不执行）

```bash
plink2 --vcf outputs_04b/demo.vcf --geno 0.10 --mind 0.10 --maf 0.05 --export vcf --out demo_qc
java -jar beagle.jar gt=demo_qc.vcf out=demo_imputed
```

**R代码**（外部工具代码模板，本 notebook 不执行）

```r
library(rMVP)
library(bigmemory)
MVP.Data(fileVCF="demo_imputed.vcf.gz",
         filePhe="outputs_04b/demo_trait.tsv", sep.phe="\t",
         fileKin=TRUE, filePC=TRUE, out="mvp")
genotype <- attach.big.matrix("mvp.geno.desc")
phenotype <- read.table("mvp.phe", header=TRUE)
map <- read.table("mvp.geno.map", header=TRUE)
Kinship <- attach.big.matrix("mvp.kin.desc")
imMVP <- MVP(phe=phenotype, geno=genotype, map=map, K=Kinship,
             nPC.MLM=3, method="MLM")
```

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=402
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 402


In [2]:
from pathlib import Path
from scipy.stats import chi2
root=Path('outputs_04b'); root.mkdir(exist_ok=True)
n,m=80,60; p=rng.uniform(.03,.5,m); G=rng.binomial(2,p,(n,m)).astype(float); miss=rng.random(G.shape)<.06; G[miss]=np.nan
# 把同一教学矩阵写成一个最小VCF，使后面的PLINK/Beagle模板与Python分析共享输入。
samples=[f'S{i+1}' for i in range(n)]; lines=['##fileformat=VCFv4.2','##contig=<ID=1>','##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">','\t'.join(['#CHROM','POS','ID','REF','ALT','QUAL','FILTER','INFO','FORMAT']+samples)]
for j in range(m):
 gt=[]
 for g in G[:,j]: gt.append('./.' if np.isnan(g) else ('0/0' if g==0 else ('0/1' if g==1 else '1/1')))
 lines.append('\t'.join(['1',str((j+1)*10000),f'M{j+1}','A','G','60','PASS','.','GT']+gt))
(root/'demo.vcf').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('wrote',root/'demo.vcf')
# QC指标：样本/位点缺失、MAF和HWE卡方近似
sample_missing=np.mean(np.isnan(G),axis=1); rows=[]
for j in range(m):
 x=G[:,j]; x=x[~np.isnan(x)]; af=x.mean()/2; maf=min(af,1-af); obs=np.array([(x==0).sum(),(x==1).sum(),(x==2).sum()]); exp=len(x)*np.array([(1-af)**2,2*af*(1-af),af**2]); stat=np.sum((obs-exp)**2/np.maximum(exp,1e-9)); rows.append((j,1-len(x)/n,maf,chi2.sf(stat,1)))
qc=pd.DataFrame(rows,columns=['marker','missing','MAF','HWE_p']); print('samples >10% missing',int(np.sum(sample_missing>.1))); print(qc.head().round(4).to_string(index=False))

# 简单均值填充只用于说明统计影响，不替代Beagle单倍型填充
Gimp=G.copy(); means=np.nanmean(Gimp,axis=0); ii=np.where(np.isnan(Gimp)); Gimp[ii]=means[ii[1]]
af_before=np.nansum(G,axis=0)/(2*np.sum(~np.isnan(G),axis=0)); af_after=Gimp.mean(axis=0)/2
print('max |AF before-after mean imputation|',round(float(np.max(np.abs(af_before-af_after))),6))
# 输出同一教学数据的剂量表，供迁移到rMVP前检查ID/位点顺序
pd.DataFrame(Gimp,index=samples,columns=[f'M{i+1}' for i in range(m)]).to_csv(root/'imputed_dosage.tsv',sep='\t',index_label='Taxa')
trait=.65*Gimp[:,10]+rng.normal(0,1,n)
pd.DataFrame({'Taxa':samples,'trait':trait}).to_csv(root/'demo_trait.tsv',sep='\t',index=False)
print('wrote',root/'imputed_dosage.tsv',root/'demo_trait.tsv')


wrote outputs_04b/demo.vcf
samples >10% missing 6
 marker  missing    MAF  HWE_p
      0    0.075 0.2027 0.4901
      1    0.000 0.4250 0.2434
      2    0.050 0.1513 0.8164
      3    0.050 0.1776 0.6364
      4    0.075 0.4054 0.6862
max |AF before-after mean imputation| 0.0
wrote outputs_04b/imputed_dosage.tsv outputs_04b/demo_trait.tsv


结果解释：示例VCF中6个样本缺失率超过10%；简单均值填充保持每个位点样本等位基因频率，因此max |AF before-after|为0，但这不等于填充正确——它不利用LD，压低剂量方差且未表达填充不确定性。外部命令模板仅作为真实软件迁移接口，不列为当前Python验证环境的执行结果。

可修改练习：提高随机缺失率并比较均值填充与“保持缺失”对剂量方差的影响；在安装相应软件的环境中再用同一demo.vcf跑PLINK/Beagle/rMVP并记录版本。